# Tiền xử lý dữ liệu cho dữ liệu về loại thuốc phản ứng của người bệnh

Import các thư viện sau:

numpy,
pandas

In [ ]:
import numpy as np
import pandas as pd

# Giới thiệu về dữ liệu
Giả sử bạn là một người nghiên cứu về dữ liệu y học, bạn đã thu thập được một tập dữ liệu các bệnh nhân của cùng một loại bệnh, trong khi điều trị mỗi bệnh nhân phản ứng với 1 trong 5 loại thuốc A, B, C, X và Y.

Công việc cần làm là xây dựng một mô hình nhằm tìm ra loại thuốc phù hợp cho bệnh nhân mới bị mắc loại bệnh trên. Các trường dữ liệu gồm gồm có: Age, Sex, Blood Pressur, Cholesterol, Na_to_K, Drug

Chúng ta sử dụng phần dữ liệu huấn luyện để xây dựng mô hình Random Forest và đánh giá mô hình này trên tập đánh giá, sau đó sử dụng mô hình để phán đoán cho bệnh nhân mới

# Load dữ liệu từ file
Đọc dữ liệu vào pandas dataframe

In [ ]:
my_data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Học Máy/drug200.csv")
my_data.head()

,Age,Sex,BP,Cholesterol,Na_to_K,Drug
0,23,F,HIGH,HIGH,25.355,drugY
1,47,M,LOW,HIGH,13.093,drugC
2,47,M,LOW,HIGH,10.114,drugC
3,28,F,NORMAL,HIGH,7.798,drugX
4,61,F,LOW,HIGH,18.043,drugY


Kích thước của dữ liệu

In [ ]:
my_data.shape

(200, 6)

# Tiền xử lý dữ liệu
my_data là Drug.csv sau khi đọc vào pandas dataframe, chúng ta định nghĩa các tham số sau:

1.   X as the Ma trận thuộc tính (data of my_data)
2.   y as the Vector thuốc phản hồi (target)



In [ ]:
X = my_data[['Age','Sex','BP','Cholesterol','Na_to_K']].values
X[0:5]

array([[23, 'F', 'HIGH', 'HIGH', 25.355],
       [47, 'M', 'LOW', 'HIGH', 13.093],
       [47, 'M', 'LOW', 'HIGH', 10.114],
       [28, 'F', 'NORMAL', 'HIGH', 7.798],
       [61, 'F', 'LOW', 'HIGH', 18.043]], dtype=object)

# Lưu ý
Một vài trường trong dữ liệu có dạng categorical, như Sex và BP. Mô hình Random Forest trong thư viện này chỉ xử lý với dữ liệu số thực do đó chúng ta cần chuyển các thuộc tính trên về dạng số thực tương ứng. Sử dụng skearn.preprocessing để chuyển dữ liệu dạng categorical về số thực

In [ ]:
from sklearn import preprocessing


le_sex = preprocessing.LabelEncoder()
le_sex.fit(['F','M']) # thư viện fit từ data dạng nhãn sang dạng số vd F-> 0 , M -> 1
X[:,1] = le_sex.transform(X[:,1]) # thư viện chuyển data trong bảng từ dạng ban đầu sang dạng đã mã hóa

le_bp = preprocessing.LabelEncoder()
le_bp.fit(['HIGH','LOW','NORMAL'])
X[:,2] = le_bp.transform(X[:,2])

le_Chol = preprocessing.LabelEncoder()
le_Chol.fit([ 'NORMAL', 'HIGH'])
X[:,3] = le_Chol.transform(X[:,3])

X[0:10]

array([[23, 0, 0, 0, 25.355],
       [47, 1, 1, 0, 13.093],
       [47, 1, 1, 0, 10.114],
       [28, 0, 2, 0, 7.798],
       [61, 0, 1, 0, 18.043],
       [22, 0, 2, 0, 8.607],
       [49, 0, 2, 0, 16.275],
       [41, 1, 1, 0, 11.037],
       [60, 1, 2, 0, 15.171],
       [43, 1, 1, 1, 19.368]], dtype=object)

# Chuẩn hóa thuộc tính Na_to_K về mean=0 và std=1
Để tránh việc xuất hiện 1 loại dữ liệu đột biến về kích cỡ làm cho quá trình huấn luyện bị sai lệch

In [ ]:
mean = X[:, -1].mean()
std = X[:, -1].std()
X[:, -1] = (X[:, -1] - mean) / std
X[0:5]

array([[23, 0, 0, 0, 1.2865221173753503],
       [47, 1, 1, 0, -0.4151453955143358],
       [47, 1, 1, 0, -0.8285581765368749],
       [28, 0, 2, 0, -1.1499626749753444],
       [61, 0, 1, 0, 0.2717942708373152]], dtype=object)

# Dữ liệu target về loại thuốc phản ứng

In [ ]:
y = my_data["Drug"]
print(y[0:5])

0    drugY
1    drugC
2    drugC
3    drugX
4    drugY
Name: Drug, dtype: object


# Đưa y về dạng số thực

In [ ]:
le_sex = preprocessing.LabelEncoder()
le_sex.fit(['drugA','drugB', 'drugC', 'drugX', 'drugY'])
y = le_sex.transform(y)

print(y[0:5])

[4 2 2 3 4]
